In [1]:
import math

In [2]:
def f(x):
    return 3*x**2 - 4*x + 5

In [3]:
f(2.0)

9.0

In [4]:
h = 0.0001
x = 2.0

slope = (f(x+h) - f(x)) / h
slope

8.000300000023941

In [5]:
for h in [0.1 , 0.01, 0.001, 0.0001, 0.00001, 0.000001]:
    slope = (f(x+h) - f(x)) / h
    print(h, slope)

0.1 8.3
0.01 8.02999999999976
0.001 8.003000000000426
0.0001 8.000300000023941
1e-05 8.000030000054892
1e-06 8.000003001384925


In [6]:
def f(a, b, c):
    return a*b + c

a, b, c = 2.0 , -3.0, 10.0
h = 0.0001

# gradient of a
grad_a = (f(a+h, b, c) - f(a, b, c)) / h

# gradient of b
grad_b = (f(a, b+h, c) - f(a, b, c)) / h

# gradient of c
grad_c = (f(a, b, c+h) - f(a, b, c)) / h

print("grad_a:", grad_a)
print("grad_b:", grad_b)
print("grad_c:", grad_c)

grad_a: -3.000000000010772
grad_b: 2.0000000000042206
grad_c: 0.9999999999976694


In [7]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

In [8]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c
print(d)
print(d._prev)
print(d._op)

Value(data=4.0)
{Value(data=10.0), Value(data=-6.0)}
+


In [9]:
print(e)
print(e._prev)
print(e._op)

Value(data=-6.0)
{Value(data=-3.0), Value(data=2.0)}
*


In [10]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [11]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c

d.grad = 1.0           # we start from the root, the derivative according to itself = 1
d._backward()          # distribute the gradient of d to e and c
e._backward()          # distribute the gradient of e to a and b

print("a.grad:", a.grad)
print("b.grad:", b.grad)
print("c.grad:", c.grad)
print("e.grad:", e.grad)

a.grad: -3.0
b.grad: 2.0
c.grad: 1.0
e.grad: 1.0


In [12]:
def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
        node._backward()

Value.backward = backward

In [13]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c

d.backward()

print("a.grad:", a.grad)
print("b.grad:", b.grad)
print("c.grad:", c.grad)

a.grad: -3.0
b.grad: 2.0
c.grad: 1.0


In [14]:
a = Value(3.0)
b = a + a

b.backward()

print("a.grad:", a.grad)

a.grad: 2.0


In [15]:
x = Value(0.0)
y = x.tanh()
print(y)

Value(data=0.0)


In [16]:
# inputs
x1 = Value(2.0)
x2 = Value(0.0)

# weights
w1 = Value(-3.0)
w2 = Value(1.0)

# bias
b = Value(6.8813735870195432)

# forward pass
x1w1 = x1 * w1
x2w2 = x2 * w2
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b          # This value is usually called as "pre-activation"
o = n.tanh()              # the version that has passed through activation

print("n:", n)
print("o:", o)

n: Value(data=0.8813735870195432)
o: Value(data=0.7071067811865476)


In [17]:
o.backward()

print("x1.grad:", x1.grad)
print("x2.grad:", x2.grad)
print("w1.grad:", w1.grad)
print("w2.grad:", w2.grad)
print("b.grad:", b.grad)

x1.grad: -1.4999999999999996
x2.grad: 0.4999999999999999
w1.grad: 0.9999999999999998
w2.grad: 0.0
b.grad: 0.4999999999999999


In [18]:
# the establishment of the neuron from scratch
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)

x1w1 = x1 * w1
x2w2 = x2 * w2
n = x1w1 + x2w2 + b
o = n.tanh()

print("inital o:", o)

# target and loss
target = Value(1.0)
loss = (o - target) * (o - target)

print("loss:", loss)

inital o: Value(data=0.7071067811865476)
loss: Value(data=0.08578643762690492)


In [19]:
loss.backward()

print("w1.grad:", w1.grad)
print("w2.grad:", w2.grad)
print("b.grad:", b.grad)

w1.grad: -0.5857864376269047
w2.grad: 0.0
b.grad: -0.29289321881345237


In [20]:
learning_rate = 0.05

w1.data -= learning_rate * w1.grad
w2.data -= learning_rate * w2.grad
b.data -= learning_rate * b.grad

# run the forward pass again with the updated weights
x1w1 = x1 * w1
x2w2 = x2 * w2
n = x1w1 + x2w2 + b
o = n.tanh()

new_loss = (o - target) * (o - target)

print("new o:", o)
print("new loss:", new_loss)

new o: Value(data=0.7418570930536283)
new loss: Value(data=0.06663776040672309)


In [21]:
# establish neuon from scratch
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
target = Value(1.0)

learning_rate = 0.05

for i in range(20):
    # forward pass
    x1w1 = x1 * w1
    x2w2 = x2 * w2
    n = x1w1 + x2w2 + b
    o = n.tanh()
    loss = (o - target) * (o - target)

    # reset gradients (clear the remaining accumulation from the previous round)
    w1.grad = 0
    w2.grad = 0
    b.grad = 0

    # backward pass
    loss.backward()

    # update
    w1.data -= learning_rate * w1.grad
    w2.data -= learning_rate * w2.grad
    b.data -= learning_rate * b.grad

    print(f"iter {i}: loss={loss.data:.6f}, o={o.data:.6f}")

iter 0: loss=0.085786, o=0.707107
iter 1: loss=0.066638, o=0.741857
iter 2: loss=0.054359, o=0.766849
iter 3: loss=0.045832, o=0.785915
iter 4: loss=0.039574, o=0.801067
iter 5: loss=0.034791, o=0.813476
iter 6: loss=0.031019, o=0.823877
iter 7: loss=0.027971, o=0.832754
iter 8: loss=0.025458, o=0.840445
iter 9: loss=0.023351, o=0.847190
iter 10: loss=0.021560, o=0.853168
iter 11: loss=0.020019, o=0.858512
iter 12: loss=0.018680, o=0.863326
iter 13: loss=0.017505, o=0.867692
iter 14: loss=0.016467, o=0.871675
iter 15: loss=0.015543, o=0.875327
iter 16: loss=0.014716, o=0.878691
iter 17: loss=0.013970, o=0.881804
iter 18: loss=0.013296, o=0.884694
iter 19: loss=0.012682, o=0.887386


In [22]:
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b = Value(6.8813735870195432)
target = Value(1.0)

learning_rate = 2.0 # the only line that changed

for i in range(20):
    x1w1 = x1 * w1
    x2w2 = x2 * w2
    n = x1w1 + x2w2 + b
    o = n.tanh()
    loss = (o - target) * (o - target)

    w1.grad = 0
    w2.grad = 0
    b.grad = 0

    loss.backward()

    # update
    w1.data -= learning_rate * w1.grad
    w2.data -= learning_rate * w2.grad
    b.data -= learning_rate * b.grad

    print(f"iter {i}: loss={loss.data:.6f}, o={o.data:.6f}")

iter 0: loss=0.085786, o=0.707107
iter 1: loss=0.000001, o=0.999020
iter 2: loss=0.000001, o=0.999020
iter 3: loss=0.000001, o=0.999020
iter 4: loss=0.000001, o=0.999020
iter 5: loss=0.000001, o=0.999020
iter 6: loss=0.000001, o=0.999020
iter 7: loss=0.000001, o=0.999020
iter 8: loss=0.000001, o=0.999021
iter 9: loss=0.000001, o=0.999021
iter 10: loss=0.000001, o=0.999021
iter 11: loss=0.000001, o=0.999021
iter 12: loss=0.000001, o=0.999021
iter 13: loss=0.000001, o=0.999021
iter 14: loss=0.000001, o=0.999021
iter 15: loss=0.000001, o=0.999021
iter 16: loss=0.000001, o=0.999021
iter 17: loss=0.000001, o=0.999021
iter 18: loss=0.000001, o=0.999021
iter 19: loss=0.000001, o=0.999021


In [23]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1)) 

    def __call__(self, x):
        # multiply each input by its weight, sum them up, and add the bias
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [24]:
n = Neuron(2)
x = [Value(2.0), Value(3.0)]
print(n(x))

Value(data=-0.4010785479975656)


In [25]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [26]:
l = Layer(2, 3)
x = [Value(2.0), Value(3.0)]
print(l(x))

[Value(data=-0.9847666685477036), Value(data=0.9890852890910854), Value(data=0.9151807906266277)]


In [27]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers =[Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [28]:
mlp = MLP(2, [4, 4, 1])
x = [Value(2.0), Value(3.0)]
print(mlp(x))

[Value(data=0.7247557999315357)]


In [29]:
# mini dataset
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]   # target output for each x sample

# a 3-input, 4->4->1 tiered MLP setup
mlp = MLP(3, [4, 4, 1])

# forward pass: give all 4 samples to the model 
ypred = [mlp(x)[0] for x in xs]
print(ypred)

[Value(data=0.5550747172905605), Value(data=0.34291940713508806), Value(data=0.6080554949726338), Value(data=0.5619040993073325)]


In [30]:
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
print(loss)

Value(data=4.779161534369533)


In [31]:
loss.backward()

In [32]:
print(mlp.layers[0].neurons[0].w[0].grad)

4.025738179478466


In [33]:
params = mlp.parameters()
print(len(params))

41


In [34]:
for k in range(20):
    # forward pass
    ypred= [mlp(x)[0] for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # reset all gradients
    for p in mlp.parameters():
        p.grad = 0

    # backward pass
    loss.backward()

    # update
    learning_rate = 0.05
    for p in mlp.parameters():
        p.data -= learning_rate * p.grad

    print(f"iter {k}: loss={loss.data:.6f}")

iter 0: loss=4.779162
iter 1: loss=2.764554
iter 2: loss=2.458701
iter 3: loss=2.197128
iter 4: loss=1.786563
iter 5: loss=1.146457
iter 6: loss=0.582966
iter 7: loss=0.339557
iter 8: loss=0.234008
iter 9: loss=0.177314
iter 10: loss=0.142284
iter 11: loss=0.118591
iter 12: loss=0.101535
iter 13: loss=0.088685
iter 14: loss=0.078665
iter 15: loss=0.070638
iter 16: loss=0.064065
iter 17: loss=0.058587
iter 18: loss=0.053952
iter 19: loss=0.049980


In [35]:
ypred = [mlp(x)[0] for x in xs]
for y, yp in zip(ys, ypred):
    print(f"target: {y}, estimation: {yp.data:.4f}")

target: 1.0, estimation: 0.8712
target: -1.0, estimation: -0.9530
target: -1.0, estimation: -0.8687
target: 1.0, estimation: 0.8975
